# 🌌 ExoModel — Quickstart

Turn your Python objects into LLM-powered agents.

This notebook covers the four core capabilities:

1. **Create & update** — populate object fields from natural language
2. **RAG grounding** — attach documents so the object reasons within your business context
3. **Agentic tools** — expose methods with `@llm_function` and let the agent call them autonomously
4. **Collections** — generate and update lists of objects in one LLM call

**Provider:** Google Gemini — [get a free API key at Google AI Studio](https://aistudio.google.com/apikey)

---
## Before you start — Open this in Google Colab

Google Colab is a free Python environment that runs entirely in the browser. No installation on your machine required.

**To create a new project:**

1. Go to [colab.research.google.com](https://colab.research.google.com)
2. Sign in with a Google account
3. Click **File → New notebook** — this creates a blank project
4. Copy each code cell from this guide into your notebook and run them in order with **Shift + Enter**

> **Tip:** if this notebook is already open in Colab (e.g. via a GitHub link), skip steps 1–4 and just run the cells from top to bottom.

---

## Step 1 — Install

> **Note:** after the installation completes, pip may show a message starting with `ERROR: pip's dependency resolver...`. This is a version conflict warning with other packages pre-installed in Colab — it does **not** affect ExoModel. Run the next cell to confirm everything is working.

In [ ]:
!pip install "exomodel[google]" -q

In [ ]:
import exomodel
print(f"ExoModel {exomodel.__version__} installed successfully.")

## Step 2 — Configure your API Key

1. Click the 🔑 **Secrets** icon in the left sidebar
2. Click **Add new secret**
3. Set the name to `GOOGLE_API_KEY` and paste your key as the value
4. Enable the **Notebook access** toggle — without this, the notebook cannot read the secret

Then run the cell below.

If you're running this notebook locally, just enter the key when prompted.

In [ ]:
import os

try:
    from google.colab import userdata
    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
except Exception:
    from getpass import getpass
    os.environ["GOOGLE_API_KEY"] = getpass("Enter your Google API key: ")

os.environ["MY_LLM_MODEL"] = "google_genai:gemini-2.5-flash-lite"
os.environ["MY_EMB_MODEL"] = "google_genai:gemini-embedding-001"

print("Ready.")

---
## Part 1 — Create and Update Objects from Natural Language

Inherit from `ExoModel`, define your fields, and call `.create()` with a plain-English description.
The object populates its own fields — no parsers, no manual mapping.

> **Expected warnings on first run:**
> - `LangChainPendingDeprecationWarning` — an internal LangGraph notice about a future default change. Safe to ignore.
> - `WARNING: Mode 'hybrid' requested without RAG context. Falling back to 'generalist'.` — this class has no documents attached, so ExoModel switches to general LLM knowledge automatically. Expected behavior.

In [ ]:
from exomodel import ExoModel

class Proposal(ExoModel):
    client: str = ""
    project_description: str = ""
    budget: float = 0.0
    timeline_weeks: int = 0

# Populate from a single natural-language instruction
p = Proposal.create(
    "Draft a proposal for Tesla to build a real-time charging analytics dashboard. "
    "Budget around $80k, delivery in 12 weeks."
)

print(p.get_instance_json())

In [ ]:
# Update specific fields from a new instruction — only changed fields are overwritten
p.update_object("Increase the budget by 10% and extend the timeline to 14 weeks")

print(p.get_instance_json())

---
## Part 2 — RAG Grounding

Override `get_rag_sources()` to attach documents to your class.
The object uses them as a knowledge base when populating fields and running analysis.
Sources can be local files, PDFs, or URLs.

In [ ]:
import os

rules = """# Proposal Rules
- Minimum project value is $10,000.
- Every proposal must include a 10% safety margin in the budget.
- We do not work with companies in the tobacco industry.
- All proposals over $50,000 must be flagged for legal review.
"""

RULES_PATH = os.path.abspath("proposal_rules.md")

with open(RULES_PATH, "w") as f:
    f.write(rules)

print(f"Rules file created at: {RULES_PATH}")

In [ ]:
class GroundedProposal(ExoModel):
    client: str = ""
    project_description: str = ""
    budget: float = 0.0
    timeline_weeks: int = 0
    safety_margin_applied: bool = False
    flagged_for_legal: bool = False

    @classmethod
    def get_rag_sources(cls):
        return [RULES_PATH]

# The object is aware of the rules when it populates itself
p2 = GroundedProposal.create(
    "Draft a proposal for Acme Corp for a cloud data warehouse migration. "
    "Budget around $75k, 10 weeks."
)

print(p2.get_instance_json())

In [ ]:
# The object checks itself against the rules document and reports findings
analysis = p2.run_analysis()
print(analysis)

---
## Part 3 — Agentic Tools with `@llm_function`

Decorate any method with `@llm_function`. When you call `master_prompt()`,
ExoAgent reads the instruction, picks the right method, and calls it.
No `if/else`, no intent routing by hand.

In [ ]:
from exomodel import ExoModel, llm_function

class LeadContact(ExoModel):
    name: str = ""
    company: str = ""
    budget_signal: str = ""
    status: str = "new"
    disqualification_reason: str = ""

    @llm_function
    def qualify(self):
        """Qualify this lead — mark them as a strong prospect worth pursuing."""
        self.status = "qualified"

    @llm_function
    def disqualify(self, reason: str):
        """Disqualify this lead and record the specific reason."""
        self.status = "disqualified"
        self.disqualification_reason = reason

    @llm_function
    def request_more_info(self):
        """Mark this lead as needing more information before a decision can be made."""
        self.status = "pending_info"

# Create a lead from raw notes
lead = LeadContact.create(
    "Sarah Connor from Skynet Inc. Mentioned a budget of around $5k. Wants a simple brochure site."
)

print("Before master_prompt:")
print(lead.get_instance_json())

In [ ]:
# ExoAgent reads the instruction and calls the right method autonomously.
# The prompt should name the intended action — the agent handles argument extraction and dispatch.
lead.master_prompt("Disqualify this lead. The budget of $5k is too low for our minimum project value.")

print("After master_prompt:")
print(lead.get_instance_json())

---
## Part 4 — Collections with `ExoModelList`

Generate and update a typed list of objects in a single LLM call.
Export to CSV when done.

In [ ]:
from exomodel import ExoModel, ExoModelList

class ProjectTask(ExoModel):
    title: str = ""
    owner: str = ""
    estimated_hours: int = 0
    priority: str = "medium"

class ProjectPlan(ExoModelList[ProjectTask]):
    pass

# Generate a full task list from one instruction
plan = ProjectPlan()
plan.create_list(
    "Break down a website redesign project into 5 tasks. "
    "Assign owners from: Alice, Bob, Carlos."
)

for task in plan.items:
    print(f"{task.title} | {task.owner} | {task.priority} | {task.estimated_hours}h")

In [ ]:
# Update all items in one LLM call
plan.update_list("Mark all tasks as high priority")

for task in plan.items:
    print(f"{task.title} | {task.owner} | {task.priority}")

In [ ]:
# Export to CSV — ready for a spreadsheet, database, or API
print(plan.to_csv())

---
## What's Next

- [Official Documentation](https://exomodel.ai)
- [GitHub Repository](https://github.com/exomodel-ai/exomodel)
- [PyPI](https://pypi.org/project/exomodel/)
- [Contact](mailto:contact@exomodel.ai)

**Install for your project:**

```bash
pip install "exomodel[google]"      # Gemini
pip install "exomodel[anthropic]"   # Claude
pip install "exomodel[openai]"      # OpenAI
pip install "exomodel[all]"         # all providers
```